# CyberShield: AI/ML Phishing Detection Model Training
### Final Year Engineering Project - Department of Information Technology

This notebook trains and benchmarks **four supervised Machine Learning models** on phishing detection corpora using Natural Language Processing (NLP) with **TF-IDF vectorization**:
1. **Multinomial Naive Bayes (MNB)**
2. **Logistic Regression (LR)**
3. **Random Forest Classifier (RF)**
4. **Linear Support Vector Machine (Linear SVC)**

In [ ]:
# Step 1: Install & Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Libraries successfully imported!")

## 2. Load the Benchmark Phishing Dataset

In [ ]:
# Load dataset (Legitimate vs Phishing)
url = "https://raw.githubusercontent.com/datasets/phishing/master/phishing_dataset.csv"
# Fallback local or inline demo dataset
data = {
    'text': [
        'URGENT: Your PayPal account has been suspended due to unauthorized activity. Please verify your login credentials immediately.',
        'Dear customer, your bank account access is restricted. Click here to confirm your identity and billing details.',
        'Security Alert: We detected an unusual signin attempt on your Microsoft account. Reset your password now.',
        'Congratulations! You have won $5,000,000 in the international online lottery. Claim your reward by providing wire transfer details.',
        'Your Netflix subscription payment failed. Update your credit card and billing information to prevent service interruption.',
        'Federal Tax Refund Notification: You are eligible for an immediate tax refund. Submit your SSN and bank details to claim payment.',
        'Final Notice: Overdue invoice #98213 is attached. Pay immediately via bitcoin wallet to prevent legal action.',
        'Dear employee, IT support requires you to re-authenticate your corporate email access. Enter your password to continue.',
        'Immediate action required: Suspicious wire transaction detected on your debit card. Call our helpline or click to cancel.',
        'Your package delivery is on hold due to missing address details. Pay $2.99 re-delivery fee to release the parcel.',
        'Hi team, please find attached the meeting minutes and action items from today sprint planning session.',
        'Reminder: The weekly engineering team sync has been rescheduled to Thursday at 3:00 PM in Conference Room B.',
        'Hello Alice, could you please review the attached draft proposal for the new client project and provide feedback?',
        'Good morning, here is the updated documentation link for the microservice deployment pipeline on GitHub.',
        'Hi all, the office will remain closed on Friday for the public holiday. Have a wonderful long weekend!',
        'Dear team, please submit your quarterly performance self-assessment forms by the end of next week.',
        'Hello Bob, thank you for sharing the financial report. I have reviewed the budget figures and left comments.',
        'Team, the automated integration test suite completed successfully on the main branch. Release build is ready for QA testing.',
        'Hi everyone, please welcome Sarah to our software development team! She will be working on backend microservices.',
        'Reminder: Annual open enrollment for health insurance benefits begins next Monday. Check the HR portal for plan details.'
    ],
    'label': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
}
df = pd.DataFrame(data)
print(f"Total Samples: {len(df)}")
print(df['label'].value_counts())
df.head()

## 3. NLP Feature Engineering (TF-IDF Vectorization)

In [ ]:
# Split Train / Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.25, random_state=42, stratify=df['label'])

# Fit TF-IDF Vectorizer with Unigrams and Bigrams
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=1500)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())} tokens")

## 4. Train and Compare Models

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=100, random_state=42),
    'Linear Support Vector Classifier': LinearSVC(random_state=42)
}

metrics_data = []
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    metrics_data.append({
        'Algorithm': name,
        'Accuracy (%)': round(acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'F1-Score (%)': round(f1 * 100, 2)
    })

results_df = pd.DataFrame(metrics_data)
results_df

## 5. Visualizations for Final Year Project Report

In [ ]:
# Plot Accuracy Comparison Bar Chart
plt.figure(figsize=(10, 5))
sns.barplot(x='Algorithm', y='Accuracy (%)', data=results_df, palette='viridis')
plt.title('Machine Learning Algorithm Performance Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, 105)
for index, row in results_df.iterrows():
    plt.text(index, row['Accuracy (%)'] + 2, f"{row['Accuracy (%)']}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot Confusion Matrix for Random Forest
rf_model = models['Random Forest Classifier']
rf_preds = rf_model.predict(X_test_vec)
cm = confusion_matrix(y_test, rf_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Legitimate (0)', 'Phishing (1)'], 
            yticklabels=['Legitimate (0)', 'Phishing (1)'])
plt.xlabel('Predicted Class', fontweight='bold')
plt.ylabel('Actual Class', fontweight='bold')
plt.title('Random Forest Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()